In [1]:
import pandas as pd
import numpy as np
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")

In [2]:
ke_df=pd.read_excel("Buffalo_NY_OB_KINGElvis.xlsx",sheet_name='Elvis_Review')
ke_df=ke_df[ke_df['INTERV_INIT']!=999]
ke_df=ke_df[ke_df['HAVE_5_MIN_FOR_SURVECode']==1]

df=pd.read_csv('elvisbuffalony2024obweekday_export_odbc.csv')

In [3]:
ke_df.head()

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,...,2x_REVIEWED_BY,2x_REVIEWED_FLAG,ADMIN_APPROVED,SURVEY_RECOVERY,SURVEY_RECOVERY_REVIEWED_BY,Recovery Check,2x_REVIEWED_BY.1,2x_REVIEWED_FLAG.1,ADMIN_APPROVED.1,RECORD_INFO
78,2024-09-25,5907,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
79,2024-09-25,5914,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80,2024-09-25,5917,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
82,2024-09-25,5920,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
83,2024-09-25,5922,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns

In [5]:
access_egres_walk_column_checks=['valaccesswalk','valegresswalk']
access_egres_walk_column=check_all_characters_present(df,access_egres_walk_column_checks)
access_egres_walk_column

['VAL_ACCESS_WALK', 'VAL_EGRESS_WALK']

In [6]:
ke_df=pd.merge(ke_df,df[['id',*access_egres_walk_column]],on='id',how='left',indicator=True)
ke_df['Matched'] = (ke_df['_merge'] == 'both').astype(int)
ke_df.drop(columns=['_merge'],inplace=True)

In [7]:
# ke_df[(ke_df[access_egres_walk_column[0]==1])&(ke_df[access_egres_walk_column[1]==1])]
condition1 = ke_df[access_egres_walk_column[0]] == 1
condition2 = ke_df[access_egres_walk_column[1]] == 1
result = ke_df[condition1 & condition2]


In [8]:
result

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,...,SURVEY_RECOVERY,SURVEY_RECOVERY_REVIEWED_BY,Recovery Check,2x_REVIEWED_BY.1,2x_REVIEWED_FLAG.1,ADMIN_APPROVED.1,RECORD_INFO,VAL_ACCESS_WALK,VAL_EGRESS_WALK,Matched
0,2024-09-25,5907,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1
1,2024-09-25,5914,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1
2,2024-09-25,5917,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1
3,2024-09-25,5920,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1
4,2024-09-25,5922,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5794,2024-11-08,16343,HereAPI,Aditya,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1
5795,2024-11-08,16344,HereAPI,Aditya,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1
5796,2024-11-08,16345,HereAPI,Aditya,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1
5798,2024-11-08,16347,HereAPI,Aditya,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1


In [9]:
condition1 = ke_df[access_egres_walk_column[0]] != 1
condition2 = ke_df[access_egres_walk_column[1]] != 1
result = ke_df[condition1 | condition2]
result

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,...,SURVEY_RECOVERY,SURVEY_RECOVERY_REVIEWED_BY,Recovery Check,2x_REVIEWED_BY.1,2x_REVIEWED_FLAG.1,ADMIN_APPROVED.1,RECORD_INFO,VAL_ACCESS_WALK,VAL_EGRESS_WALK,Matched
8,2024-09-25,5930,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,1
14,2024-09-25,5954,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,1
63,2024-09-25,6112,HereAPI,Priyanka,Remove,NaN,O=D,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,1
77,2024-09-25,6136,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,1
78,2024-09-25,6137,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5781,2024-11-08,16314,HereAPI,Aditya,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,1
5785,2024-11-08,16327,HereAPI,Aditya,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,1
5787,2024-11-08,16330,HereAPI,Aditya,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,1
5797,2024-11-08,16346,HereAPI,Aditya,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,1
